# Clone Repository and set up the Environment

In [1]:
!pwd

/content


In [2]:
!git clone https://github.com/VictoryChianumba/robust-eeg-models

fatal: destination path 'robust-eeg-models' already exists and is not an empty directory.


In [3]:
%cd robust-eeg-models

/content/robust-eeg-models


In [4]:
!git config --global user.email "chianumbav@gmial.com"
!git config --global user.name "Victory Chianumba"

In [14]:
# 1️⃣ Upgrade the package manager
!pip install --upgrade --quiet pip

!pip install torch torchvision torchaudio
!pip install mne moabb
!pip install torch-lr-finder
!pip install optuna

# Clean uninstall
!pip uninstall -y braindecode

# Install latest code from GitHub (which includes CTNet)
!pip install git+https://github.com/braindecode/braindecode.git@master --no-cache-dir

Found existing installation: braindecode 0.8.1
Uninstalling braindecode-0.8.1:
  Successfully uninstalled braindecode-0.8.1
  Cloning https://github.com/braindecode/braindecode.git (to revision master) to /tmp/pip-req-build-812kojuw
  Running command git clone --filter=blob:none --quiet https://github.com/braindecode/braindecode.git /tmp/pip-req-build-812kojuw
  Resolved https://github.com/braindecode/braindecode.git to commit 2abb2dd0b5cdba9be0309d4f128f0b2138946fc0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for braindecode: filename=braindecode-0.8.1-py3-none-any.whl size=282928 sha256=74fb7c9a61fb32b2349dc398970953422c2c3be87d76efe69b47912242e6241b
  Stored in directory: /tmp/pip-ephem-wheel-cache-ytbgdyqk/wheels/b6/8b/2b/0da876924d16f36b5bdd43797902e55d90426ed71fd51642f2
Successfully built braindecode


In [6]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available ✅")


Braindecode version: 0.8.1
CTNet is available ✅


In [7]:
import torch
import importlib
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

import numpy as np
import os
import sys

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


In [8]:
from braindecode.models import EEGNetv4, CTNet
from braindecode.datasets import MOABBDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler


# Loading data for training

In [9]:

dataset = MOABBDataset(dataset_name="BNCI2014001", subject_ids=[1])

/usr/local/lib/python3.11/dist-packages/moabb/datasets/download.py:56: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))


MNE_DATA is not already configured. It will be set to default location in the home directory - /root/mne_data
All datasets will be downloaded to this location, if anything is already downloaded, please move manually to this location


/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 42.8M/42.8M [00:00<00:00, 66.4GB/s]
SHA256 hash of downloaded file: 054f02e70cf9c4ada1517e9b9864f45407939c1062c6793516585c6f511d0325
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 43.8M/43.8M [

In [10]:
from braindecode.preprocessing import Preprocessor,exponential_moving_standardize, preprocess


low_cut_hz = 4.0  # low cut frequency for filtering
high_cut_hz = 38.0  # high cut frequency for filtering
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000

preprocessors = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
    Preprocessor(
        lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
        factor=1e6,
    ),
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
    Preprocessor(
        exponential_moving_standardize,  # Exponential moving standardization
        factor_new=factor_new,
        init_block_size=init_block_size,
    ),
]

# Preprocess the data
preprocess(dataset, preprocessors, n_jobs=-1)

/usr/local/lib/python3.11/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


In [11]:
from braindecode.preprocessing import create_windows_from_events

trial_start_offset_seconds = -0.5
# Extract sampling frequency, check that they are same in all datasets
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
# Calculate the window start offset in samples.
trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

# Create windows using braindecode function for this. It needs parameters to
# define how windows should be used.
windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=trial_start_offset_samples,
    trial_stop_offset_samples=0,
    preload=True,
)


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']


In [12]:
splitted = windows_dataset.split("session")
train_set = splitted["0train"]  # Session train
test_set = splitted["1test"]  # Session evaluation

In [15]:
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

n_classes = len(torch.unique(torch.tensor([sample[1] for sample in windows_dataset])))
classes = list(range(n_classes))
# Extract number of chans and time steps from dataset
n_channels = windows_dataset[0][0].shape[0]
n_times = windows_dataset[0][0].shape[1]

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,

)

# Display torchinfo table describing the model
print(model)

# Send model to GPU
if cuda:
    model.cuda()

/usr/local/lib/python3.11/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  return F.conv2d(


Layer (type (var_name):depth-idx)                                           Input Shape               Output Shape              Param #                   Kernel Shape
CTNet (CTNet)                                                               [1, 22, 1125]             [1, 4]                    --                        64
├─Rearrange (ensuredim): 1-1                                                [1, 22, 1125]             [1, 1, 22, 1125]          --                        --
├─_PatchEmbeddingEEGNet (cnn): 1-2                                          [1, 1, 22, 1125]          [1, 17, 40]               --                        --
│    └─Sequential (eegnet_module): 2-1                                      [1, 1, 22, 1125]          [1, 40, 1, 17]            --                        --
│    │    └─Conv2d (0): 3-1                                                 [1, 1, 22, 1125]          [1, 20, 22, 1125]         1,280                     [1, 64]
│    │    └─BatchNorm2d (1): 3-2           

In [16]:
from sklearn.model_selection import train_test_split
from skorch.helper import SliceDataset, predefined_split
from torch.utils.data import Subset

X_train = SliceDataset(train_set, idx=0)
y_train = np.array([y for y in SliceDataset(train_set, idx=1)])
train_indices, val_indices = train_test_split(
    X_train.indices_, test_size=0.2, shuffle=False
)
train_subset = Subset(train_set, train_indices)
val_subset = Subset(train_set, val_indices)

In [17]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, cross_val_score
from skorch.callbacks import LRScheduler
from braindecode.models import CTNet

from braindecode import EEGClassifier

lr = 0.0625 * 0.01
weight_decay = 0
batch_size = 64
n_epochs = 200

train_val_split = KFold(n_splits=5, shuffle=False)
clf = EEGClassifier(
    model,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.AdamW,
    train_split=None,
    optimizer__lr=lr,
    optimizer__weight_decay=weight_decay,
    batch_size=batch_size,
    callbacks=[
        "accuracy",
        ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
    ],
    device=device,
    classes=classes,
    max_epochs=n_epochs,
)

param_grid = {
    "optimizer__lr": [0.00625, 0.000625],
    "optimizer__weight_decay": [0, 0.0005],
    "batch_size": [64, 128],
}

# By setting n_jobs=-1, grid search is performed
# with all the processors, in this case the output of the training
# process is not printed sequentially
search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=train_val_split,
    return_train_score=True,
    scoring="accuracy",
    refit=True,
    verbose=1,
    error_score="raise",
    n_jobs=1,
)

search.fit(X_train, y_train)
search_results = pd.DataFrame(search.cv_results_)

best_run = search_results[search_results["rank_test_score"] == 1].squeeze()

best_parameters = best_run["params"]

Streaming output truncated to the last 5000 lines.
     49            1.0000        0.1579  0.0005  0.1828
     50            1.0000        0.1221  0.0005  0.1321
     51            1.0000        0.1247  0.0005  0.1311
     52            1.0000        0.1518  0.0005  0.1378
     53            1.0000        0.1424  0.0005  0.1306
     54            1.0000        0.1375  0.0005  0.1388
     55            1.0000        0.1778  0.0005  0.1370
     56            1.0000        0.1413  0.0005  0.1360
     57            1.0000        0.1159  0.0005  0.1318
     58            1.0000        0.0981  0.0005  0.1354
     59            1.0000        0.1104  0.0005  0.1314
     60            1.0000        0.1296  0.0005  0.1321
     61            1.0000        0.1315  0.0005  0.1314
     62            1.0000        0.1499  0.0005  0.1346
     63            1.0000        0.1085  0.0005  0.1356
     64            1.0000        0.1286  0.0005  0.1380
     65            1.0000        0.1381  0.0005  0.13

In [18]:
from braindecode import EEGClassifier

# Extract best parameters from GridSearch
best_params = search.best_params_

# Create new classifier with best parameters
final_clf = EEGClassifier(
    model,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.AdamW,
    train_split=None,  # Use all training data
    optimizer__lr=best_params['optimizer__lr'],  # Use best lr other parameters from best_params
    optimizer__weight_decay=best_params['optimizer__weight_decay'],
    batch_size=best_params['batch_size'],  # Use best batch size
    callbacks=[
        "accuracy",
        ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
    ],
    device=device,
    classes=classes,
    max_epochs=n_epochs,
)

# Train on full training set
final_clf.fit(train_set, y=None)

# evaluated the model after training
y_test = test_set.get_metadata().target
test_acc = final_clf.score(test_set, y=y_test)
print(f"Test acc: {(test_acc * 100):.2f}%")

  epoch    train_accuracy    train_loss      lr     dur
-------  ----------------  ------------  ------  ------
      1            0.3576        3.7627  0.0063  0.5057
      2            0.2569        1.7163  0.0062  0.5022
      3            0.5694        1.3938  0.0062  0.5612
      4            0.7049        1.2187  0.0062  0.2175
      5            0.8819        1.0168  0.0062  0.1834
      6            0.8576        0.7343  0.0062  0.1922
      7            0.9201        0.5731  0.0062  0.1734
      8            0.8611        0.4898  0.0062  0.1707
      9            0.9688        0.3771  0.0062  0.1774
     10            0.9514        0.3729  0.0062  0.1895
     11            0.9028        0.3350  0.0062  0.1706
     12            0.9965        0.3228  0.0062  0.1752
     13            0.9896        0.3026  0.0062  0.1780
     14            0.9931        0.3058  0.0062  0.1815
     15            0.9896        0.1823  0.0062  0.1718
     16            0.9896        0.2220  0.0062 

In [ ]:
train_val_split = KFold(n_splits=5, shuffle=False)
# By setting n_jobs=-1, cross-validation is performed
# with all the processors, in this case the output of the training
# process is not printed sequentially
cv_results = cross_val_score(
    clf, X_train, y_train, scoring="accuracy", cv=train_val_split, n_jobs=1
)
print(
    f"Validation accuracy: {np.mean(cv_results * 100):.2f}"
    f"+-{np.std(cv_results * 100):.2f}%"
)


In [ ]:
EEGNetv4??

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

def plot_k_fold(ax, cv, all_dataset, X_train, y_train, test_set):
    """Create a sample plot for training, validation, testing."""
    bd_cmap = [
        "#3A6190",
        "#683E00",
        "#2196F3",
        "#DDF2FF",
    ]

    ax.barh("Original\nDataset", len(all_dataset), left=0, height=0.5, color=bd_cmap[0])

    # Generate the training/validation/testing data fraction visualizations for each CV split
    for ii, (tr_idx, val_idx) in enumerate(cv.split(X=X_train, y=y_train)):
        n_train, n_val, n_test = len(tr_idx), len(val_idx), len(test_set)
        n_train2 = n_train + n_val - max(val_idx) - 1
        ax.barh("cv" + str(ii + 1), min(val_idx), left=0, height=0.5, color=bd_cmap[1])
        ax.barh(
            "cv" + str(ii + 1), n_val, left=min(val_idx), height=0.5, color=bd_cmap[2]
        )
        ax.barh(
            "cv" + str(ii + 1),
            n_train2,
            left=max(val_idx) + 1,
            height=0.5,
            color=bd_cmap[1],
        )
        ax.barh(
            "cv" + str(ii + 1),
            n_test,
            left=n_train + n_val,
            height=0.5,
            color=bd_cmap[3],
        )

    ax.invert_yaxis()
    ax.set_xlim([-int(0.1 * len(all_dataset)), int(1.1 * len(all_dataset))])
    ax.set(xlabel="Number of samples.", title="KFold Train-Test-Valid split")
    ax.legend(
        [Patch(color=bd_cmap[i]) for i in range(4)],
        ["Original set", "Training set", "Validation set", "Testing set"],
        loc="lower center",
        ncols=2,
    )
    ax.text(
        -0.07,
        0.45,
        "Train-Valid-Test split",
        rotation=90,
        verticalalignment="center",
        horizontalalignment="left",
        transform=ax.transAxes,
    )
    return ax


fig, ax = plt.subplots(figsize=(15, 7))
plot_k_fold(
    ax,
    cv=train_val_split,
    all_dataset=windows_dataset,
    X_train=X_train,
    y_train=y_train,
    test_set=test_set,
)

First, let's remove the existing cloned repository and change back to the root directory.

After running these cells, please try running the cell that imports `CTNet` again (cell `nsMZ3JXEjfYP`).